# Retail Sales & Inventory — Data Profiling

The goal of this notebook is to understand the raw data before cleaning or changing it.

We will check:
- the files and columns
- number of rows and columns
- missing values
- duplicate records and IDs
- data types
- basic statistics
- unique values in categorical columns
- dates
- negative and zero values
- simple relationships between tables

In [1]:
import pandas as pd
import os

# Path to the raw data
base = "C:/Users/selin/anaconda_projects/data science/Retail-Sales-Inventory-Project/data/raw"

print("Files in dataset:")
print(os.listdir(base))

Files in dataset:
['calendar.csv', 'data_dictionary.csv', 'inventory.csv', 'products.csv', 'sales.csv', 'stores.csv']


In [2]:
# Load the raw CSV files
sales = pd.read_csv(f"{base}/sales.csv")
products = pd.read_csv(f"{base}/products.csv")
stores = pd.read_csv(f"{base}/stores.csv")
inventory = pd.read_csv(f"{base}/inventory.csv")

datasets = {
    "Sales": sales,
    "Products": products,
    "Stores": stores,
    "Inventory": inventory
}

for name, df in datasets.items():
    print(f"\n{name} columns:")
    print(df.columns.tolist())


Sales columns:
['Sale_ID', 'Date', 'Store_ID', 'Product_ID', 'Units']

Products columns:
['Product_ID', 'Product_Name', 'Product_Category', 'Product_Cost', 'Product_Price']

Stores columns:
['Store_ID', 'Store_Name', 'Store_City', 'Store_Location', 'Store_Open_Date']

Inventory columns:
['Store_ID', 'Product_ID', 'Stock_On_Hand']


In [3]:
# Basic structure, missing values, duplicates and data types
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")

    missing = df.isnull().sum()
    print("Missing values:")
    print(missing[missing > 0])

    print("\nData types:")
    print(df.dtypes)


--- Sales ---
Rows: 829,262
Columns: 5
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

Data types:
Sale_ID        int64
Date          object
Store_ID       int64
Product_ID     int64
Units          int64
dtype: object

--- Products ---
Rows: 35
Columns: 5
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

Data types:
Product_ID           int64
Product_Name        object
Product_Category    object
Product_Cost        object
Product_Price       object
dtype: object

--- Stores ---
Rows: 50
Columns: 5
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

Data types:
Store_ID            int64
Store_Name         object
Store_City         object
Store_Location     object
Store_Open_Date    object
dtype: object

--- Inventory ---
Rows: 1,593
Columns: 3
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

Data types:
Store_ID         int64
Product_ID       int64
Stock_On_Hand    int64
dtype: object


In [4]:
# Basic statistics for numeric columns
for name, df in datasets.items():
    print(f"\n--- {name} numerical summary ---")
    display(df.describe().T)


--- Sales numerical summary ---


,count,mean,std,min,25%,50%,75%,max
Sale_ID,829262.0,414631.500000,239387.463803,1.0,207316.25,414631.5,621946.75,829262.0
Store_ID,829262.0,25.277034,14.352573,1.0,13.00,26.0,38.00,50.0
Product_ID,829262.0,15.014149,9.869417,1.0,6.00,14.0,24.00,35.0
Units,829262.0,1.315103,0.830701,1.0,1.00,1.0,1.00,30.0



--- Products numerical summary ---


,count,mean,std,min,25%,50%,75%,max
Product_ID,35.0,18.0,10.246951,1.0,9.5,18.0,26.5,35.0



--- Stores numerical summary ---


,count,mean,std,min,25%,50%,75%,max
Store_ID,50.0,25.5,14.57738,1.0,13.25,25.5,37.75,50.0



--- Inventory numerical summary ---


,count,mean,std,min,25%,50%,75%,max
Store_ID,1593.0,25.326428,14.477592,1.0,13.0,25.0,38.0,50.0
Product_ID,1593.0,18.073446,10.064254,1.0,9.0,18.0,27.0,35.0
Stock_On_Hand,1593.0,18.670433,18.997652,0.0,6.0,13.0,24.0,139.0


In [5]:
# Look at unique values in text/categorical columns
for name, df in datasets.items():
    print(f"\n--- {name} categorical columns ---")
    for col in df.select_dtypes(include="object").columns:
        print(f"\n{col}: {df[col].nunique():,} unique values")
        print(df[col].value_counts().head(10))


--- Sales categorical columns ---

Date: 638 unique values
Date
2023-04-30    3101
2023-04-29    2464
2023-03-30    2461
2022-12-24    2450
2023-06-17    2416
2023-01-06    2390
2022-04-30    2363
2023-06-24    2351
2023-05-20    2345
2023-04-08    2276
Name: count, dtype: int64

--- Products categorical columns ---

Product_Name: 35 unique values
Product_Name
Action Figure         1
PlayDoh Toolkit       1
Mini Ping Pong Set    1
Monopoly              1
Mr. Potatohead        1
Nerf Gun              1
PlayDoh Can           1
PlayDoh Playset       1
Playfoam              1
Magic Sand            1
Name: count, dtype: int64

Product_Category: 5 unique values
Product_Category
Toys                 9
Art & Crafts         8
Games                8
Sports & Outdoors    7
Electronics          3
Name: count, dtype: int64

Product_Cost: 16 unique values
Product_Cost
$3.99      5
$9.99      4
$10.99     3
$8.99      3
$13.99     3
$1.99      2
$7.99      2
$6.99      2
$11.99     2
$14.99     2
Na

In [6]:
# Check whether important IDs are duplicated
print("Duplicate Product IDs:", products["Product_ID"].duplicated().sum())
print("Duplicate Store IDs:", stores["Store_ID"].duplicated().sum())
print(
    "Duplicate Store + Product combinations:",
    inventory.duplicated(subset=["Store_ID", "Product_ID"]).sum()
)

Duplicate Product IDs: 0
Duplicate Store IDs: 0
Duplicate Store + Product combinations: 0


In [7]:
# Check the sales dates without changing the original data
dates = pd.to_datetime(sales["Date"], errors="coerce")

print("Earliest date:", dates.min())
print("Latest date:", dates.max())
print("Invalid dates:", dates.isna().sum())

Earliest date: 2022-01-01 00:00:00
Latest date: 2023-09-30 00:00:00
Invalid dates: 0


In [8]:
# Check for negative and zero values

print("Negative units:", (sales["Units"] < 0).sum())
print("Zero units:", (sales["Units"] == 0).sum())

# Check the data type before checking prices and costs
print("\nProduct Price data type:", products["Product_Price"].dtype)
print("Product Cost data type:", products["Product_Cost"].dtype)

# Stock is already numeric
print("\nNegative stock:", (inventory["Stock_On_Hand"] < 0).sum())
print("Zero stock:", (inventory["Stock_On_Hand"] == 0).sum())

Negative units: 0
Zero units: 0

Product Price data type: object
Product Cost data type: object

Negative stock: 0
Zero stock: 77


In [9]:
# Check whether every Product_ID in Sales exists in Products
missing_products = sales[~sales["Product_ID"].isin(products["Product_ID"])]
print("Sales records with a missing Product_ID in Products:", len(missing_products))

# Check whether every Store_ID in Sales exists in Stores
missing_stores = sales[~sales["Store_ID"].isin(stores["Store_ID"])]
print("Sales records with a missing Store_ID in Stores:", len(missing_stores))

Sales records with a missing Product_ID in Products: 0
Sales records with a missing Store_ID in Stores: 0


## Profiling Summary

After running the checks above, record the main findings here in simple language.

Example:
- The dataset contains four tables: Sales, Products, Stores and Inventory.
- I checked the number of rows and columns in each table.
- I checked for missing values and duplicate records.
- I checked the data types and basic numerical statistics.
- I checked important IDs for duplicates.
- I checked the sales date range and invalid dates.
- I checked for negative and zero values. But found a difference in data type between prices and costs which will be attended to in data cleaning
- I checked whether Sales Product_IDs and Store_IDs exist in the related tables.

### Next step
Use the findings from this notebook to decide what needs to be cleaned in `02_data_cleaning.ipynb`. Do not clean or modify the raw data in this profiling notebook.